<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Reflection, Parallelisierung und Hierarchie
</b></font> </br></p>

---


**Beitrag zum Leitprojekt:** Für den PMO-Assistenten sind drei Muster über den einfachen Supervisor hinaus wichtig: **Reflection** prüft Antworten vor der Ausgabe und schickt unbelegte Entwürfe zur Überarbeitung zurück (**Prüfen**), **Parallelisierung** lässt unabhängige Recherchen gleichzeitig laufen (**Handeln**), und **Hierarchie** ordnet mehrere Teams, wenn ein Supervisor die Fachbreite nicht mehr abdeckt. Reflection und Parallelisierung zahlen sich schon in kleinen Systemen aus; Hierarchie ist optional und erst bei ausreichender fachlicher oder organisatorischer Komplexität sinnvoll.

**Praxis-Transfer:** Mara Vogt erhält Briefings, deren Antworten vor der Ausgabe auf Quellenbelege geprüft werden, deren Recherchen parallel laufen und deren Teams bei wachsender Komplexität klar getrennt sind.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "m21-reflection-parallel-hierarchy"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    load_prompt,
    show_trace,
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS


# 1 | Übersicht
---

Dieses Notebook behandelt drei Muster, die den einfachen Supervisor aus M20 erweitern: Reflection verbessert die Qualität einer Antwort, Parallelisierung verkürzt die Laufzeit, Hierarchie ordnet größere Systeme.

| Muster | Kernidee | Nutzen | Risiko | Wann einsetzen |
|---|---|---|---|---|
| Reflection (Evaluator-Optimizer) | Ein Agent schreibt, ein zweiter prüft; Überarbeitung bis zur Freigabe oder zum Limit | Höhere Qualität, Quellenpflicht | Mehr Modellaufrufe, Endlosschleife ohne Limit | Wenn Antwortqualität wichtiger ist als Tempo |
| Parallelisierung (Fan-out/Fan-in) | Unabhängige Worker laufen gleichzeitig, die Ergebnisse werden zusammengeführt | Kürzere Laufzeit | Reducer nötig, abhängige Teams gehen nicht | Bei unabhängigen Teilaufgaben |
| Hierarchie | Teams als Subgraphen mit privatem State | Wiederverwendbare Teams, klare Grenzen | Struktur-Overhead | Erst wenn ein Supervisor zu viele Worker steuert |


<p><font color='darkblue' size="4">
ℹ️ <b>Information:</b>
</font></p>

Die Reihenfolge entspricht dem Nutzen: Reflection und Parallelisierung lohnen sich schon in kleinen Systemen. Hierarchie wird für Mara erst relevant, wenn der PMO-Assistent mehrere interne Teams braucht, etwa Quellenarbeit, Antwortsynthese und Qualitätsprüfung. Eine einzelne Tabellenfrage oder eine einfache Statusabfrage kommt mit einem flachen Supervisor aus.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Drei Muster im PMO-Assistent</font> </br></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart LR
    subgraph REF["Reflection"]
        direction TB
        EN["Entwurf"] --> KR["Kritik"]
        KR -->|abgelehnt| EN
        KR -->|freigegeben| FG["Freigabe"]
    end

    subgraph PAR["Parallelisierung"]
        direction TB
        VT["Verteilen"] --> T1["Team Tabellen"]
        VT --> T2["Team Protokoll"]
        T1 --> ZF["Zusammenführen"]
        T2 --> ZF
    end

    subgraph HIE["Hierarchie"]
        direction TB
        HG["Hauptgraph"] --> QT["Quellen-Team"]
        QT --> ST["Synthese-Team"]
    end

    %% Zuweisung
    class EN,VT,HG input;
    class KR metric;
    class FG,ZF output;

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
'''

mermaid(diagram, width=980)


# 2 | Reflection (Evaluator-Optimizer)
---

Bei Reflection erzeugt ein Agent einen Entwurf (Generator), ein zweiter bewertet ihn nach festen Kriterien (Evaluator) und nennt bei Ablehnung konkret, was fehlt. Der Generator überarbeitet den Entwurf, bis der Evaluator freigibt oder das Iterationslimit erreicht ist.

| Rolle | Aufgabe | Modell |
|---|---|---|
| `entwurf_node` | Antwort schreiben bzw. anhand der Kritik überarbeiten | `WORKER` |
| `kritik_node` | Entwurf prüfen, strukturierte Entscheidung `freigegeben` + `begruendung` | `JUDGE` |
| `route_nach_kritik` | Nur Freigabe oder Limit beendet die Schleife | kein Modell |

Die Kriterien müssen **prüfbar** sein, zum Beispiel „endet mit `Quelle: <Dateiname>`“. Vage Kriterien wie „gute Antwort“ führen zu milden Kritikern, die alles freigeben.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Reflection-Schleife</font> </br></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart LR
    START([START]) --> ENT["Entwurf
WORKER"]
    ENT --> KRI["Kritik
JUDGE"]
    KRI -->|freigegeben| END([END])
    KRI -->|abgelehnt und Runden unter Limit| ENT
    KRI -->|Limit erreicht| END

    %% Zuweisung
    class START input;
    class KRI metric;
    class END output;

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
'''

mermaid(diagram, width=820)


### 2.1 | Umsetzung mit LLM

Der Graph verwendet zwei Modelle: den günstigeren `WORKER` als Generator und den stärkeren `JUDGE` als Kritiker. Der Kritiker liefert über `with_structured_output(Kritik)` eine maschinenlesbare Entscheidung, der Router gibt `Literal["ueberarbeiten", "fertig"]` zurück. `MAX_REVISIONS` schützt vor Endlosschleifen, zusätzlich begrenzt `recursion_limit` den Graphen.

Der Entwurf wird in der ersten Runde bewusst ohne Quellenpflicht angefordert; der Kritiker prüft sie. ⚠️ Modelle sind nicht deterministisch: Die Anzahl der Runden kann zwischen Läufen variieren.


In [ ]:
from typing import Literal, TypedDict
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END

MAX_REVISIONS = 3

KONTEXT = (
    "entscheidungsprotokoll_gesamt.pdf: D1 (lokale Vektordatenbank) gilt nur für den Prototyp. "
    "D2 (Cloud-Vektordatenbank) gilt für den produktiven Betrieb.\n"
    "protokoll_steuerkreis_2026-04-14.pdf: Der Steuerkreis hat D2 am 14.04.2026 bestätigt."
)


class Kritik(BaseModel):
    freigegeben: bool = Field(description="True nur, wenn alle drei Kriterien erfüllt sind")
    begruendung: str = Field(description="Kurze Begründung; bei Ablehnung: was konkret fehlt")


class ReflectionState(TypedDict):
    query: str
    draft: str
    feedback: str
    revisions: int
    approved: bool
    log: list[str]


writer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Du bist Briefing-Writer im PMO-Assistenten. Beantworte die Frage in höchstens zwei Sätzen "
               "ausschließlich anhand des Kontexts.\n\nKontext:\n{kontext}"),
    ("human", "Frage: {query}\n\nBisheriger Entwurf: {draft}\nKritik des Reviewers: {feedback}\n\n"
              "Falls eine Kritik vorliegt, behebe sie im neuen Entwurf."),
])
kritik_prompt = ChatPromptTemplate.from_messages([
    ("system", "Du bist Quality Reviewer. Prüfe den Entwurf streng nach drei Kriterien:\n"
               "1. Jede Aussage ist durch den Kontext belegt.\n"
               "2. Der Entwurf endet mit 'Quelle: <Dateiname>' (Dateiname aus dem Kontext).\n"
               "3. Höchstens zwei Sätze vor dem Quellenhinweis.\n"
               "Gib freigegeben=true nur, wenn alle drei Kriterien erfüllt sind. Nenne bei Ablehnung konkret, was fehlt.\n\n"
               "Kontext:\n{kontext}"),
    ("human", "Frage: {query}\n\nEntwurf: {draft}"),
])

writer_chain = writer_prompt | init_chat_model(WORKER) | StrOutputParser()
kritiker = kritik_prompt | init_chat_model(JUDGE).with_structured_output(Kritik)


def entwurf_node(state: ReflectionState, config: RunnableConfig) -> dict:
    draft = writer_chain.invoke(
        {"kontext": KONTEXT, "query": state["query"], "draft": state["draft"] or "–", "feedback": state["feedback"] or "–"},
        config=config,
    )
    runde = state["revisions"] + 1
    return {"draft": draft, "revisions": runde, "log": state["log"] + [f"Runde {runde}: Entwurf erstellt"]}


def kritik_node(state: ReflectionState, config: RunnableConfig) -> dict:
    kritik = kritiker.invoke({"kontext": KONTEXT, "query": state["query"], "draft": state["draft"]}, config=config)
    status = "freigegeben" if kritik.freigegeben else "abgelehnt"
    return {
        "approved": kritik.freigegeben,
        "feedback": "" if kritik.freigegeben else kritik.begruendung,
        "log": state["log"] + [f"Runde {state['revisions']}: {status} – {kritik.begruendung}"],
    }


def route_nach_kritik(state: ReflectionState) -> Literal["ueberarbeiten", "fertig"]:
    if state["approved"] or state["revisions"] >= MAX_REVISIONS:
        return "fertig"
    return "ueberarbeiten"


reflection_builder = StateGraph(ReflectionState)
reflection_builder.add_node("entwurf_node", entwurf_node)
reflection_builder.add_node("kritik_node", kritik_node)
reflection_builder.add_edge(START, "entwurf_node")
reflection_builder.add_edge("entwurf_node", "kritik_node")
reflection_builder.add_conditional_edges(
    "kritik_node", route_nach_kritik, {"ueberarbeiten": "entwurf_node", "fertig": END}
)
reflection_agent = reflection_builder.compile()
print("Reflection-Graph kompiliert.")


In [ ]:
run_cfg = {"run_name": "m21_kap2_reflection", "tags": ["m21", "reflection"], "recursion_limit": 15}

reflection_result = reflection_agent.invoke(
    {
        "query": "Welche Vektordatenbank gilt im produktiven Betrieb?",
        "draft": "",
        "feedback": "",
        "revisions": 0,
        "approved": False,
        "log": [],
    },
    config=run_cfg,
)

mprint("**Reflection-Log:**")
mprint("\n".join(f"- {eintrag}" for eintrag in reflection_result["log"]))

mprint("**Ergebnis:**")
if reflection_result["approved"]:
    print(reflection_result["draft"])
else:
    print("Keine freigegebene Antwort (Limit erreicht).")
print("Runden:", reflection_result["revisions"], "| freigegeben:", reflection_result["approved"])


### 2.2 | Stolperfallen

| Problem | Gegenmaßnahme |
|---|---|
| Kritiker zu mild: gibt alles frei | Prüfbare Kriterien, stärkeres Modell (`JUDGE`) als Kritiker |
| Endlosschleife | `MAX_REVISIONS` und `recursion_limit` |
| Kosten wachsen mit jeder Runde | Limit klein halten (2–3), günstigeren Generator verwenden |
| Generator ignoriert die Kritik | Kritik explizit im Prompt übergeben, wie in `entwurf_node` |

Das Kritik-Gate aus M20 ist dasselbe Prinzip innerhalb eines Supervisors; hier steht die Schleife allein im Mittelpunkt.


# 3 | Parallelisierung
---

Sind Teilaufgaben voneinander unabhängig, müssen sie nicht nacheinander laufen. Bei der Parallelisierung verteilt ein Node die Arbeit (Fan-out), die Worker laufen im selben Schritt, und ein Synthese-Node führt die Ergebnisse zusammen (Fan-in).


### 3.1 | Fan-out/Fan-in mit Send

Hier arbeiten Tabellen- und Protokollrecherche **gleichzeitig**:

1. Eine Routing-Funktion gibt pro Team ein `Send(node, eingabe)` zurück — LangGraph startet alle Nodes im selben Schritt.
2. Jeder Node schreibt sein Ergebnis in ein Feld mit **Reducer** (`Annotated[list[str], operator.add]`). Ohne Reducer würde der letzte Wert die anderen überschreiben.
3. Ein Synthese-Node läuft erst, wenn alle Teams fertig sind.

⚠️ Parallel lohnt sich nur bei unabhängigen Teams. Hängt ein Team vom Ergebnis des anderen ab, bleibt die sequenzielle Reihenfolge richtig.


In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send


class ParallelState(TypedDict):
    query: str
    teams: list[str]
    befunde: Annotated[list[str], operator.add]
    answer: str


class TeamEingabe(TypedDict):
    team: str
    query: str


def verteilen_node(state: ParallelState) -> dict:
    return {"teams": ["tabellen", "protokoll"]}


def fan_out(state: ParallelState) -> list[Send]:
    return [Send("team_node", {"team": team, "query": state["query"]}) for team in state["teams"]]


def team_node(state: TeamEingabe) -> dict:
    befunde = {
        "tabellen": "Tabelle: 4 von 5 Aufgaben erledigt",
        "protokoll": "Protokoll: D2 gilt für den produktiven Betrieb",
    }
    return {"befunde": [befunde[state["team"]]]}


def zusammenfuehren_node(state: ParallelState) -> dict:
    return {"answer": " | ".join(sorted(state["befunde"]))}


parallel_builder = StateGraph(ParallelState)
parallel_builder.add_node("verteilen_node", verteilen_node)
parallel_builder.add_node("team_node", team_node)
parallel_builder.add_node("zusammenfuehren_node", zusammenfuehren_node)
parallel_builder.add_edge(START, "verteilen_node")
parallel_builder.add_conditional_edges("verteilen_node", fan_out, ["team_node"])
parallel_builder.add_edge("team_node", "zusammenfuehren_node")
parallel_builder.add_edge("zusammenfuehren_node", END)
parallel_agent = parallel_builder.compile()

parallel_result = parallel_agent.invoke(
    {"query": "Wie ist der Projektstand?", "teams": [], "befunde": [], "answer": ""},
    config={"run_name": "m21_kap3_parallel", "tags": ["m21", "parallel"]},
)
print("Befunde:", len(parallel_result["befunde"]))
print(parallel_result["answer"])


# 4 | Hierarchie
---

### 4.1 | Wann reicht ein Supervisor nicht mehr?

Hierarchie ist im Kern ein Supervisor über Supervisoren. Sie lohnt sich erst, wenn ein Supervisor zu viele Worker steuern müsste, wenn Teams wiederverwendbar sein sollen oder wenn Teams interne Zwischendaten brauchen, die der Hauptgraph nicht sehen soll. In LangGraph dienen dafür **Subgraphen**: Ein Team ist ein eigener, kompilierter Graph, der im Hauptgraph als Node läuft.

Das Beispiel unten verwendet drei Teams (Quellen-, Synthese- und Qualitäts-Team) in fester Reihenfolge und kommt ohne Modellaufrufe aus.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image as IPImage

class HierarchyState(TypedDict):
    query: str
    source_pack: str
    synthesis_draft: str
    answer: str
    quality_status: str
    approved: bool
    log: list[str]


def log_plus(state: HierarchyState, eintrag: str) -> list[str]:
    return state.get("log", []) + [eintrag]

print("Hierarchie-State ist bereit.")


<p><font color='darkblue' size="4">
  <b>Wichtig:</b>
</font></p>

Die Hierarchie muss die Übergaben reduzieren, nicht vergrößern. Die Teams geben deshalb verdichtete Pakete zurück: Quellenpaket, Syntheseentwurf und Qualitätsstatus.

### 4.2 | Teams als Subgraphen

Der Hauptgraph ruft Quellen-, Synthese- und Qualitäts-Team in fester Reihenfolge auf; die interne Zuständigkeit bleibt jeweils im Subgraphen gekapselt. Das Beispiel bleibt ohne Modellaufrufe reproduzierbar.

**Namenskonvention**

| Konzept | Flach (Supervisor) | Hierarchisch |
|---|---|---|
| Tabellenarbeit | `tabellen_worker_node` | `tabellen_spezialist_node` im Quellen-Team |
| Textarbeit | `text_worker_node` | `fliess_text_spezialist_node` im Quellen-Team |
| Synthese | `synthese_worker_node` | `antwort_writer_node` + `citation_formatter_node` im Synthese-Team |

Der Rollenwechsel ist bewusst: In der Hierarchie sind die Worker nicht mehr direkt am Hauptgraph, sondern Teil eines Teams. Die fachliche Namensbasis bleibt gleich; nur die Hierarchieebene wird im Namen sichtbar.

**Zusätzliche Tool-Ebene im flachen Supervisor-System:** `tabellen_worker_node`/`text_worker_node` rufen intern die `@tool`-Funktionen `tabellen_signal_tool`/`text_signal_tool` per `.invoke()` auf, um den Befund-String zu erzeugen, und schreiben das Ergebnis anschließend als State-Feld (`tabellenbefund`/`textbefund`). Die `*_signal_tool`-Namen bezeichnen also eine Tool-Ebene unterhalb der Worker, keine eigene Rolle – in der Hierarchie entfällt diese Ebene, weil `tabellen_spezialist_node`/`fliess_text_spezialist_node` den Befund direkt erzeugen.


In [ ]:
def tabellen_spezialist_node(state: HierarchyState) -> dict:
    befund = "Tabelle: 4 von 5 Aufgaben im Follow-up-Tracker erledigt, 1 offen."
    return {"source_pack": befund, "log": log_plus(state, "Tabellen-Spezialist liefert Kennzahl")}


def fliess_text_spezialist_node(state: HierarchyState) -> dict:
    paket = state["source_pack"] + " Text: Quellenhinweise machen Antworten prüfbar."
    return {"source_pack": paket, "log": log_plus(state, "Fließtext-Spezialist ergänzt Argument")}


quellen_builder = StateGraph(HierarchyState)

quellen_builder.add_node("tabellen_spezialist_node", tabellen_spezialist_node)
quellen_builder.add_node("fliess_text_spezialist_node", fliess_text_spezialist_node)

quellen_builder.add_edge(START, "tabellen_spezialist_node")
quellen_builder.add_edge("tabellen_spezialist_node", "fliess_text_spezialist_node")
quellen_builder.add_edge("fliess_text_spezialist_node", END)

quellen_subgraph = quellen_builder.compile()
print("Quellen-Team kompiliert.")


In [ ]:
def antwort_writer_node(state: HierarchyState) -> dict:
    entwurf = "Antwort: " + state["source_pack"]
    return {"synthesis_draft": entwurf, "log": log_plus(state, "Antwort-Writer erstellt Entwurf")}


def citation_formatter_node(state: HierarchyState) -> dict:
    antwort = state["synthesis_draft"] + " Quelle: follow_up_tracker.pdf."
    return {"answer": antwort, "log": log_plus(state, "Citation-Formatter ergänzt Quelle")}


synthese_builder = StateGraph(HierarchyState)
synthese_builder.add_node("antwort_writer_node", antwort_writer_node)
synthese_builder.add_node("citation_formatter_node", citation_formatter_node)
synthese_builder.add_edge(START, "antwort_writer_node")
synthese_builder.add_edge("antwort_writer_node", "citation_formatter_node")
synthese_builder.add_edge("citation_formatter_node", END)

synthese_subgraph = synthese_builder.compile()

In [ ]:
def quellen_gate_node(state: HierarchyState) -> dict:
    status = "belegt" if "Quelle:" in state.get("answer", "") else "quelle_fehlt"
    return {"quality_status": status, "log": log_plus(state, f"Quellen-Gate: {status}")}


def out_of_corpus_check_node(state: HierarchyState) -> dict:
    approved = state.get("quality_status") == "belegt" and "Nicht im Korpus" not in state.get("answer", "")
    return {"approved": approved, "log": log_plus(state, f"Out-of-Corpus-Check: {'approved' if approved else 'gesperrt'}")}

qualitaet_builder = StateGraph(HierarchyState)

qualitaet_builder.add_node("quellen_gate_node", quellen_gate_node)
qualitaet_builder.add_node("out_of_corpus_check_node", out_of_corpus_check_node)
qualitaet_builder.add_edge(START, "quellen_gate_node")
qualitaet_builder.add_edge("quellen_gate_node", "out_of_corpus_check_node")
qualitaet_builder.add_edge("out_of_corpus_check_node", END)

qualitaets_subgraph = qualitaet_builder.compile()

In [ ]:
haupt_builder = StateGraph(HierarchyState)

haupt_builder.add_node("quellen_subgraph", quellen_subgraph)
haupt_builder.add_node("synthese_subgraph", synthese_subgraph)
haupt_builder.add_node("qualitaets_subgraph", qualitaets_subgraph)
haupt_builder.add_edge(START, "quellen_subgraph")
haupt_builder.add_edge("quellen_subgraph", "synthese_subgraph")
haupt_builder.add_edge("synthese_subgraph", "qualitaets_subgraph")
haupt_builder.add_edge("qualitaets_subgraph", END)

hierarchy_agent = haupt_builder.compile()

In [ ]:
display(IPImage(hierarchy_agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
run_cfg = {"run_name": "m21_kap3_hierarchie_grundlauf", "tags": ["m21", "hierarchie", "pmo-briefing"]}

hierarchie_result = hierarchy_agent.invoke(
    {
        "query": "Wie ist der Stand der Aufgaben laut Follow-up-Tracker?",
        "source_pack": "",
        "synthesis_draft": "",
        "answer": "",
        "quality_status": "",
        "approved": False,
        "log": [],
    },
    config=run_cfg,
)

mprint("**Hierarchie-Log:**")
mprint("\n".join(f"- {eintrag}" for eintrag in hierarchie_result["log"]))

mprint("**Finale Antwort:**")
print(hierarchie_result["answer"])
print("Freigegeben:", hierarchie_result["approved"])

### 4.3 | Privater Subgraph-State

Bisher teilen sich alle Teams denselben `HierarchyState`. Das funktioniert, solange der Subgraph nur Keys des Parents nutzt. Braucht ein Team **interne Hilfsdaten** (hier `raw_data`), bekommt es einen **eigenen State**. Ein **Wrapper-Node** im Parent übersetzt dann in beide Richtungen:

1. Nur die benötigten Felder aus dem Parent-State an den Subgraph übergeben (`invoke`)
2. Das Ergebnis entgegennehmen
3. Nur ausgewählte Felder zurück in den Parent-State schreiben

| Variante | Wann sinnvoll |
|---|---|
| Gemeinsame Keys, Subgraph direkt als Node | Teams arbeiten auf denselben Feldern (Abschnitt 4.2) |
| Eigener State + Wrapper-Node | Team hat private Zwischendaten oder ein abweichendes Schema |

⚠️ Bei gemeinsamen Keys liest und schreibt der Subgraph dieselben Kanäle wie der Parent. Ohne Reducer überschreibt ein Update den alten Wert. Quelle: [LangGraph-Doku: Subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs)

In [ ]:
class QuellenState(TypedDict):
    query: str
    raw_data: list[str]
    source_pack: str


def rohdaten_sammeln_node(state: QuellenState) -> dict:
    rohdaten = [
        "Tabelle 1: 4 von 5 Aufgaben erledigt",
        "Tabelle 2: 1 Aufgabe offen (Rechte- und Rollensteuerung)",
        "Tabelle 3: Latenz = 1.2 s",
    ]
    return {"raw_data": rohdaten}


def paket_verdichten_node(state: QuellenState) -> dict:
    return {"source_pack": f"{len(state['raw_data'])} Tabellen verdichtet: 4 von 5 Aufgaben erledigt, 1 offen"}


privat_builder = StateGraph(QuellenState)
privat_builder.add_node("rohdaten_sammeln_node", rohdaten_sammeln_node)
privat_builder.add_node("paket_verdichten_node", paket_verdichten_node)
privat_builder.add_edge(START, "rohdaten_sammeln_node")
privat_builder.add_edge("rohdaten_sammeln_node", "paket_verdichten_node")
privat_builder.add_edge("paket_verdichten_node", END)
privater_quellen_subgraph = privat_builder.compile()


def quellen_team_wrapper(state: HierarchyState) -> dict:
    ergebnis = privater_quellen_subgraph.invoke({"query": state["query"], "raw_data": [], "source_pack": ""})
    return {
        "source_pack": ergebnis["source_pack"],
        "log": log_plus(state, "Quellen-Team (privater State) liefert nur das Paket"),
    }


wrapper_builder = StateGraph(HierarchyState)
wrapper_builder.add_node("quellen_team_wrapper", quellen_team_wrapper)
wrapper_builder.add_edge(START, "quellen_team_wrapper")
wrapper_builder.add_edge("quellen_team_wrapper", END)
wrapper_agent = wrapper_builder.compile()

wrapper_result = wrapper_agent.invoke({"query": "Wie ist der Stand der Aufgaben laut Follow-up-Tracker?", "source_pack": "", "log": []})
print("Quellenpaket:", wrapper_result["source_pack"])
print("raw_data im Parent-State:", "raw_data" in wrapper_result)
print("Log:", wrapper_result["log"])

<p><font color='darkblue' size="4">
✨ <b>Empfehlung:</b>
</font></p>

Hierarchie ist passend, wenn mehrere Teams unabhängig intern arbeiten und nur verdichtete Pakete an die nächste Ebene geben. Bei einfachen Standardabläufen bleibt eine Pipeline oder ein flacher Supervisor günstiger und leichter prüfbar.


# 5 | Debugging mit Trace und Log
---

Bei Schleifen, parallelen Zweigen und Teams ist der Pfad wichtiger als die finale Antwort. Das Log zeigt, wie viele Reflection-Runden nötig waren, ob alle parallelen Teams geliefert haben und ob Teams und Qualitäts-Gates in der richtigen Reihenfolge liefen.

In [ ]:
#@markdown <p><font size="4" color='green'>LangSmith Trace-Analyse</font></p>

try:
    show_trace("m21-reflection-parallel-hierarchy", limit=3, show_steps=True)
except Exception as exc:
    print("Trace-Anzeige übersprungen:", type(exc).__name__, str(exc)[:180])


# 6 | Welches Muster wann?
---

| Situation | Muster | Modul |
|---|---|---|
| Der Ablauf steht fest | Pipeline | M19 |
| Der Fragetyp bestimmt den Worker | Supervisor | M20 |
| Die Antwort muss geprüft und ggf. verbessert werden | Reflection | M21, Abschnitt 2 |
| Unabhängige Teilaufgaben, Laufzeit zählt | Parallelisierung | M21, Abschnitt 3 |
| Mehrere wiederverwendbare Teams | Hierarchie | M21, Abschnitt 4 |

Die Muster lassen sich kombinieren, zum Beispiel ein Supervisor, dessen Worker parallel arbeiten und dessen Antwort durch einen Reflection-Loop geprüft wird.


# A | Aufgaben
---


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='darkblue' size="4">
✏️  <b>Notiz:</b>
</font></p>

Die Aufgabenstellungen prüfen, ob Reflection, Parallelisierung und Hierarchie jeweils fachlich begründet und nicht nur dekorativ eingesetzt werden. Jeder Selfcheck steht direkt unter der zugehörigen Umsetzung.


**Grundlagen**

Die Reflection-Schleife aus Abschnitt 2 mit einer neuen Frage ausführen. Das Ergebnis muss zeigen, wie viele Runden nötig waren und warum der Kritiker ablehnte oder freigab.


In [ ]:
# Grundlagen: Reflection-Schleife mit einer neuen Frage ausführen
# Startpunkt: reflection_agent ist oben bereits kompiliert (Abschnitt 2)

# 1. reflection_agent.invoke(...) mit einer neuen Frage aufrufen (grundlagen_result), config=run_cfg
# 2. Ausgeben: revisions, approved und das Log
# 3. Prüfen: grundlagen_result["revisions"] <= MAX_REVISIONS


**Aufbau**

Die Parallelisierung um ein drittes Team (`risiko`) erweitern. Alle drei Befunde müssen im State ankommen.


In [ ]:
# Aufbau: drittes Team parallel ergänzen
# Startpunkt: parallel_agent aus Abschnitt 3

# 1. team_node um das Team "risiko" erweitern (eigener Befund-String)
# 2. verteilen_node gibt drei Teams zurück; parallel_agent neu kompilieren
# 3. Aufrufen (aufbau_result); Prüfen: len(aufbau_result["befunde"]) == 3


**Vertiefung**

Wahlaufgabe: **A** Reflection und Parallelisierung kombinieren (die parallel gesammelten Befunde durchlaufen den Kritiker) oder **B** begründen, wann Hierarchie überdimensioniert ist und ein flaches Pattern genügt.


In [ ]:
# Vertiefung (Wahl A): Reflection nach der Parallelisierung
# Startpunkt: parallel_agent (Abschnitt 3) und kritik_node/route_nach_kritik (Abschnitt 2)

# 1. Synthese-Node der Parallelisierung schreibt draft aus den befunde
# 2. Kritik-Schleife aus Abschnitt 2 anhängen (gleiche Kriterien, MAX_REVISIONS beachten)
# 3. Aufrufen (vertiefung_result); Prüfen: approved ist True oder das Limit wurde erreicht

# Vertiefung (Wahl B): Grenze der Hierarchie begründen
# Startpunkt: hierarchie_result aus dem Grundlauf (Abschnitt 4.2)

# 1. hierarchie_grenze (str) formulieren: wann genügt ein flaches Pattern statt Hierarchie?
# 2. hierarchie_grenze und die Länge von hierarchie_result["log"] ausgeben


<p><font color='darkblue' size="4">
✨ <b>Empfehlung:</b>
</font></p>

M21 zeigt drei Erweiterungen des Supervisors. Reflection und Parallelisierung lohnen sich in den meisten Projekten früh, Hierarchie ist die Erweiterung für einen Korpus mit mehreren klar getrennten Arbeitsteams. Für eigene Projekte reicht oft ein Supervisor mit zwei Spezialisten, dessen Antworten ein Reflection-Gate prüft.


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://ralf-42.github.io/Agenten/visualisierungen/ki_agent_architektur/)
- [LangGraph](https://ralf-42.github.io/Agenten/visualisierungen/langgraph_explainer/)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Building effective agents (Anthropic)](https://www.anthropic.com/engineering/building-effective-agents)
- [Multi-Agent-Systeme](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/multi-agent-systeme.html)
- [Agenten-Kommunikationsprotokolle](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/agenten-kommunikationsprotokolle.html)
- [LangGraph Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langgraph-best-practices.html)
- [Agenten-Architekturen](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/agent-architekturen.html)
- [Skills](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/skills.html)
